# DATA-445
## Exam 1 - Part 3

### Instructions

- **Exam 1 - Part 3** is worth **96** points.
- Please submit a **.ipynb** file to Blackboard.
- **Please strive for clarity and organization.**
- **AI usage**: 
   - You may use AI exclusively for **debugging**. 
   - **Do not** use AI to generate answers.
- **Due Date: October 9, 2026, by 12:50 PM.**

## Exercise 1

Consider the `EV_Adoption_and_Range_Anxiety_Dataset.csv` datafile. This dataset contains information about potential buyers' demographics, driving habits, charging infrastructure, and attitudes toward electric vehicles (EVs). The goal is to build classification models to predict `Will_Buy_EV`. The measure of performance is the **AUC-ROC**.

**(a)** (2 points) Read the datafile and create a data-frame called `df`. Drop the `Buyer_ID` column since it is only a row identifier and carries no predictive information.

In [ ]:
import pandas as pd

df = pd.read_csv("EV_Adoption_and_Range_Anxiety_Dataset.csv")

df = df.drop(columns=["Buyer_ID"], axis=1)

df.head()

['Buyer_ID', 'Age', 'Gender', 'Annual_Income_USD', 'City_Type', 'Daily_Commute_km', 'Number_of_Cars_Owned', 'Current_Car_Type', 'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work', 'Home_Charging_Possible', 'Environmental_Concern_Level', 'Subsidy_Available', 'Range_Anxiety_Level', 'Will_Buy_EV']


**(b)** (4 points) There are three boolean-like columns: `Home_Charging_Possible`, `Subsidy_Available`, and `Will_Buy_EV` (`Yes`/`No`). Convert `No` to 0 and `Yes` to 1.

In [16]:
columns = ['Home_Charging_Possible', 'Subsidy_Available', 'Will_Buy_EV']

df[columns] = df[columns].replace({'No': 0, 'Yes': 1})


**(c)** (8 points) There are four object columns: `Gender`, `City_Type`, `Current_Car_Type`, and `Range_Anxiety_Level`. Convert those labels into numbers. *Hint*: you may consider using `OrdinalEncoder` from `sklearn.preprocessing`.

In [17]:
from sklearn.preprocessing import OrdinalEncoder

columns = ['Gender', 'City_Type', 'Current_Car_Type', 'Range_Anxiety_Level']
df[columns] = OrdinalEncoder().fit_transform(df[columns])


**(d)** (4 points) Define `Will_Buy_EV` as the target variable, and the other variables as the input features. Also, define the 5-fold stratified cross-validation strategy (use `StratifiedKFold` with `shuffle=True` and `random_state=42`). *Note*: `Annual_Income_USD`, `Daily_Commute_km`, and `Environmental_Concern_Level` contain missing values. In parts (e) to (i), impute them with the median using `SimpleImputer` inside a `Pipeline`, so that the imputation is learned only from the training folds.

In [20]:
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

X = df.drop(columns=['Will_Buy_EV', 'Buyer_ID'])
y = df['Will_Buy_EV']

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('model', LogisticRegression(max_iter=1000)),
])


**(e)** (15 points) Train and evaluate the `LogisticRegression` model with an L2 penalty over the 5-fold cross-validation strategy defined in part (d). Ensure to standardize the inputs and optimize the value of `C` first and then run the cross-validation experiment. Use the AUC-ROC as measure of performance.

**(f)** (15 points) Train and evaluate the `RandomForestClassifier` model (`n_estimators=1000`, `max_depth=10`, and `random_state=42`) over the 5-fold cross-validation strategy defined in part (d). Use the AUC-ROC as measure of performance.

In [27]:
import numpy as np
import sklearn.ensemble
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

model = sklearn.ensemble.RandomForestClassifier(n_estimators=1000, max_depth=10, random_state=42, n_jobs=-1)
print(type(model))
randomf = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('model', model),
])

scores = cross_val_score(randomf, X, y, cv=cv, scoring="roc_auc")

print("AUC-ROC per fold:", np.round(scores, 4))
print(f"Mean AUC-ROC: {scores.mean():.4f} (± {scores.std():.4f})")

<class 'sklearn.ensemble._forest.RandomForestClassifier'>
AUC-ROC per fold: [0.9116 0.8864 0.9061 0.8991 0.8889]
Mean AUC-ROC: 0.8984 (± 0.0097)


**(g)** (15 points) Train and evaluate the `MLPClassifier` with two hidden layers: the first one with 8 neurons and `ReLU` as the activation function, the second one with 8 neurons and `ReLU` as the activation function. Also use `solver='adam'`, `max_iter=200` and `batch_size=256`, over the 5-fold cross-validation strategy defined in part (d). Use the AUC-ROC as measure of performance. Ensure to standardize the input features before you build the model.

In [30]:

from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

mlp = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', MLPClassifier(
        hidden_layer_sizes=(8, 8),
        activation='relu',
        max_iter=200,
        solver="adam",   
        batch_size=256,      
        random_state=42
)),
])


scores = cross_val_score(mlp, X, y, cv=cv, scoring="roc_auc")

print("AUC-ROC per fold:", np.round(scores, 4))
print(f"Mean AUC-ROC: {scores.mean():.4f} (± {scores.std():.4f})")

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


AUC-ROC per fold: [0.9148 0.8924 0.9058 0.8987 0.8863]
Mean AUC-ROC: 0.8996 (± 0.0100)


**(h)** (15 points) Train and evaluate the `LogisticRegression` model with an elastic-net penalty (`penalty='elasticnet'`, `solver='saga'`) over the 5-fold cross-validation strategy defined in part (d). Ensure to standardize the inputs and optimize the values of `C` and `l1_ratio` first and then run the cross-validation experiment. Use the AUC-ROC as measure of performance.

In [ ]:

from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer

pipe = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler(),
   LogisticRegression(penalty ='elasticnet', solver='saga', max_iter=5000),
)


**(i)** (15 points) Train and evaluate the `GradientBoostingClassifier` model (`n_estimators=1000`, `max_depth=10`, and `random_state=42`) over the 5-fold cross-validation strategy defined in part (d). Use the AUC-ROC as measure of performance.

In [ ]:

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer

gradientb = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler(),
    GradientBoostingClassifier(n_estimators=1000, random_state=42, max_depth=10),
)

scores = cross_val_score(gradientb, X, y, cv=cv, scoring='roc_auc')
print("Gradient Boosting Classifier Scores:", scores)
print("Mean Gradient Boosting Classifier Score:", scores.mean())

**(j)** (3 points) Using the results from parts 1(e) to 1(i), what model would you use to predict `Will_Buy_EV`?

_Your answer here._